# 02 · Causal graph and falsification (Section 4.1)
Defines the 33-edge graph (and its 27-edge variant) and tests each against every project with DoWhy's permutation-based falsification (`gcm.falsify`).

**Test.** Conditional independences are tested with a partial-correlation test on ranks (a Spearman partial correlation); kernel tests were impractical at these sample sizes. The local Markov condition (LMC) is checked for every node against each non-descendant, given its parents.

**Decision rule.** The graph is *informative* if p_TPA ≤ 0.05 (few node permutations lie in its Markov equivalence class) and *rejected* if p_LMC > 0.05 (permuted graphs violate no more independences than the proposed graph does).

In [ ]:
# On Google Colab, uncomment the next line first:
# !pip install -q dowhy==0.14 statsmodels networkx
import warnings, numpy as np, pandas as pd, networkx as nx, statsmodels.api as sm
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

# Data source: Song & Minku (TSE 2022) replication package. Set BASE to a local folder to work offline.
BASE = "https://raw.githubusercontent.com/sunnysong14/ContinualPerformanceValidityTSE2022/main/data/"
Y, T = "days_to_first_fix", "author_date_unix_timestamp"

PROJECTS = {"brackets": "Brackets", "broadleaf": "Broadleaf", "camel": "Camel", "corefx": "CoreFX",
            "django": "Django", "fabric8": "Fabric8", "jgroup": "JGroup", "nova": "Nova", "rails": "Rails",
            "rust": "Rust", "tensorflow": "TensorFlow", "vscode": "VSCode", "wp-calypso": "WP-Calypso"}
# Projects on which the 33-edge graph is informative and not rejected (Section 4.1)
COMPATIBLE = ["brackets", "camel", "corefx", "django", "fabric8", "nova",
              "rails", "rust", "tensorflow", "vscode", "wp-calypso"]
METRICS = [T, "exp", "rexp", "fix", "ns", "nf", "sexp", "lt", "nuc", "ndev", "age", "la", "ld", "entropy", Y]


def load(name, horizon_q=0.9, return_info=False):
    """Defect-inducing commits of one project, with the right-censored tail removed.

    A commit made within `horizon` of the last commit in the data has had less time to be fixed than
    the project's 90th-percentile defect lifetime, so it is excluded (Section 2.2).
    """
    raw = pd.read_csv(f"{BASE}{name}.csv")
    raw = raw.loc[:, ~raw.columns.str.startswith("Unnamed")].rename(columns={"entrophy": "entropy"})
    bug = raw[raw.contains_bug == 1]
    horizon_days = bug[Y].quantile(horizon_q)
    kept = bug[bug[T] <= bug[T].max() - horizon_days * 86400]
    df = kept.replace([np.inf, -np.inf], np.nan).dropna(subset=METRICS).reset_index(drop=True)
    df["fix"] = df["fix"].astype(float)
    if return_info:
        return df, dict(project=PROJECTS[name], all_commits=len(raw), defect_inducing=len(bug),
                        horizon_days=round(horizon_days), after_censoring=len(kept), analysed=len(df))
    return df


In [ ]:
# Revised causal graph (Fig. 2): 14 nodes, 33 edges.
DAG33_EDGES = [
    (T, "exp"), (T, "rexp"), (T, "sexp"), (T, "nuc"), (T, "ndev"), (T, Y),
    ("exp", "rexp"), ("exp", Y),
    ("rexp", "nf"), ("rexp", Y),
    ("fix", "ns"), ("fix", "nf"), ("fix", Y),
    ("ns", "nf"), ("ns", "sexp"),
    ("nf", "lt"), ("nf", "nuc"), ("nf", "la"), ("nf", "entropy"), ("nf", Y),
    ("sexp", "exp"),
    ("lt", "nuc"), ("lt", "ndev"),
    ("nuc", "ndev"), ("nuc", "age"), ("nuc", "entropy"), ("nuc", Y),
    ("ndev", "age"), ("ndev", Y),
    ("age", Y),
    ("la", "entropy"), ("la", Y),
    ("entropy", Y),
]
# 27-edge variant without `fix` and `age` (same adjustment sets)
DROP = {"fix", "age"}
DAG27_EDGES = [(a, b) for a, b in DAG33_EDGES if a not in DROP and b not in DROP]

def to_graph(edges):
    g = nx.DiGraph(edges)
    assert nx.is_directed_acyclic_graph(g)
    return g

def to_dot(edges):
    return "digraph {\n" + "\n".join(f"    {a} -> {b};" for a, b in edges) + "\n}"

G33, G27 = to_graph(DAG33_EDGES), to_graph(DAG27_EDGES)
causal_graph = to_dot(DAG33_EDGES)          # DOT string accepted by DoWhy's CausalModel
print(f"G33: {G33.number_of_nodes()} nodes, {G33.number_of_edges()} edges | "
      f"G27: {G27.number_of_nodes()} nodes, {G27.number_of_edges()} edges")


## Render the graph (optional, needs Graphviz)

In [ ]:
import subprocess, shutil
open("dag33.dot", "w").write(causal_graph.replace("digraph {", "digraph {\n    rankdir=TB; node [shape=ellipse, fontname=Helvetica];"))
if shutil.which("dot"):
    subprocess.run(["dot", "-Tpng", "-Gdpi=200", "dag33.dot", "-o", "dag33.png"], check=True)
    from IPython.display import Image; display(Image("dag33.png", width=700))

## Fast exact replica of `gcm.falsify`
DoWhy's `falsify_graph` refits every test for every permuted graph. Because every test here is a partial correlation on ranks, all tests can be computed from one rank-correlation matrix per project, which makes 1,000 permutations × 3 seeds × 13 projects feasible. The replica uses DoWhy's definitions of p_LMC and p_TPA; the next section confirms it against DoWhy itself.

In [ ]:
class RankData:
    def __init__(self, df, nodes):
        self.n = len(df); self.idx = {v: i for i, v in enumerate(nodes)}
        self.R = df[list(nodes)].rank().corr().values; self.cache = {}

    def p(self, x, y, Z):
        key = (min(x, y), max(x, y), Z)
        if key not in self.cache:
            S = [self.idx[x], self.idx[y]] + [self.idx[z] for z in sorted(Z)]
            P = np.linalg.pinv(self.R[np.ix_(S, S)])
            r = np.clip(-P[0, 1] / np.sqrt(P[0, 0] * P[1, 1]), -0.999999, 0.999999)
            z = np.sqrt(self.n - len(Z) - 3) * np.arctanh(r)
            self.cache[key] = float(2 * stats.norm.sf(abs(z)))
        return self.cache[key]

def lmc_triples(g):
    """(node, non-descendant, parents) for every local-Markov test the graph implies."""
    out = []
    for x in g.nodes:
        pa = frozenset(g.predecessors(x))
        for y in set(g.nodes) - nx.descendants(g, x) - {x} - pa:
            out.append((x, y, pa))
    return out

def violation_share(g, d, alpha=0.05):
    tr = lmc_triples(g)
    return sum(d.p(x, y, Z) <= alpha for x, y, Z in tr), len(tr)

def falsify(g, d, n_perm=1000, seed=0):
    nodes = list(g.nodes); v, n = violation_share(g, d); f = v / n
    rng = np.random.default_rng(seed); le = mec = 0
    for _ in range(n_perm):
        pm = rng.permutation(len(nodes))
        h = nx.relabel_nodes(g, {nodes[i]: nodes[j] for i, j in enumerate(pm)})
        hv, hn = violation_share(h, d); le += hv / hn <= f
        mec += all(nx.is_d_separator(g, {x}, {y}, set(Z)) for x, y, Z in lmc_triples(h))
    p_lmc, p_tpa = le / n_perm, mec / n_perm
    return dict(violations=f"{v}/{n}", p_LMC=p_lmc, p_TPA=p_tpa,
                informative=p_tpa <= 0.05, rejected=p_lmc > 0.05)

In [ ]:
SEEDS, N_PERM = (0, 1, 2), 1000
rows = []
for p in PROJECTS:
    df = load(p)
    for name, g in (("33-edge", G33), ("27-edge", G27)):
        d = RankData(df, list(g.nodes))
        res = [falsify(g, d, N_PERM, s) for s in SEEDS]
        rows.append(dict(project=PROJECTS[p], graph=name, n=len(df), violations=res[0]["violations"],
                         p_TPA=max(r["p_TPA"] for r in res),
                         p_LMC=", ".join(f"{r['p_LMC']:.3f}" for r in res),
                         verdict="not rejected" if all(r["informative"] and not r["rejected"] for r in res)
                                 else ("borderline" if any(not r["rejected"] for r in res) else "rejected")))
    print(PROJECTS[p], "done", flush=True)
F = pd.DataFrame(rows)
F.pivot(index="project", columns="graph", values="verdict")

In [ ]:
F

## Confirmation with DoWhy's own `falsify_graph`
Runs DoWhy on the rank-transformed data with the same partial-correlation test. Set `PROJECT` and `N_PERM_DOWHY`; one project with 1,000 permutations takes a few minutes.

In [ ]:
from dowhy.gcm.falsify import falsify_graph

def pcorr_p(x, y, z=None):
    x, y = np.ravel(x).astype(float), np.ravel(y).astype(float); k = 0
    if z is not None and np.size(z):
        Z = np.column_stack([np.ones(len(x)), np.reshape(z, (len(x), -1))])
        x = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]
        y = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
        k = Z.shape[1] - 1
    r = np.clip(np.corrcoef(x, y)[0, 1], -0.999999, 0.999999)
    return float(2 * stats.norm.sf(abs(np.sqrt(len(x) - k - 3) * np.arctanh(r))))

PROJECT, N_PERM_DOWHY = "corefx", 1000
np.random.seed(0)
df = load(PROJECT)
result = falsify_graph(G33, df[list(G33.nodes)].rank(), n_permutations=N_PERM_DOWHY,
                       independence_test=pcorr_p, conditional_independence_test=pcorr_p,
                       plot_histogram=False, show_progress_bar=False)
print(result)
print("informative:", result.falsifiable, "| rejected:", result.falsified)